## Objective

- Discretize continuous state space into (inventory_bucket, days_bucket) pairs
- Initialize Q-table with zeros for all state-action combinations
- Build shared evaluation framework (run_episodes helper)

## Results

- Q-table shape confirmed: (10, 10, 10) — inventory x days x actions
- Discretization function tested against sample state values
- run_episodes() validated against RandomAgent on placeholder env

## Conclusion

State discretization and evaluation infrastructure are ready for
Q-Learning training in the coming days.

# Week 2 — Q-Learning Setup

## 1. Imports

In [ ]:
import sys
sys.path.append('../src')
from q_learning_agent import QLearningAgent
from eval_utils import run_episodes
import gymnasium as gym

print("Imports successful")

## 2. Initialize Q-Learning Agent

In [ ]:
agent = QLearningAgent()
print("Q-table shape:", agent.q_table.shape)
print("Sample discretization (inv=50, days=15):",
      agent.discretize_state(50, 15))

## 3. Test Evaluation Framework

In [ ]:
from random_agent import RandomAgent

test_env = gym.make("CartPole-v1")
random_agent = RandomAgent(test_env.action_space)

results = run_episodes(random_agent, test_env, n_episodes=50)
print("Mean revenue:", results["mean_revenue"])
print("Std revenue:", results["std_revenue"])
test_env.close()

## 4. Baseline Heuristic Comparison — Random vs Fixed vs Time-Based Discount

In [ ]:
from pricing_env import PricingEnv
from baseline_agents import FixedPriceAgent, TimeBasedDiscountAgent

env = PricingEnv()

random_agent = RandomAgent(env.action_space)
fixed_agent = FixedPriceAgent(price_level_index=5)
discount_agent = TimeBasedDiscountAgent(start_price_level=9)

results = {}
results["Random"] = run_episodes(random_agent, env, n_episodes=500)
results["Fixed"] = run_episodes(fixed_agent, env, n_episodes=500)
results["Discount"] = run_episodes(discount_agent, env, n_episodes=500)

for name, r in results.items():
    print(f"{name}: mean={r['mean_revenue']:.2f}, std={r['std_revenue']:.2f}")

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(8, 5))
plt.boxplot([results[k]["all_rewards"] for k in results], tick_labels=list(results.keys()))
plt.title("Revenue Distribution — Baseline Heuristic Comparison")
plt.ylabel("Episodic Revenue")
plt.tight_layout()
plt.savefig('../outputs/baseline_comparison_boxplot.png', dpi=150)
plt.show()

## 5. Q-Learning Implementation Validation

Confirmed correct implementation:
- Bellman update: Q(s,a) += lr * (reward + gamma * max(Q(s')) - Q(s,a))
- Hyperparameters: learning_rate=0.1, discount_factor=0.95, epsilon=1.0 (decaying to 0.05)
- Epsilon-greedy action selection tested against real PricingEnv
- Reward accumulation verified via single-step test in q_learning_agent.py